# 04c — EXP_03: Sparse RAG (BM25 keyword retrieval)

**Architecture:** sparse keyword retrieval + LLM. **Retriever:** `SparseRetriever` (BM25 top-5 over chunk text). **Generator:** `llama-3.3-70b-versatile` via Groq · T=0 · k=5 · max_tokens=700. **Disk-cached** — restarts are free.

Same three-stage gating as EXP_01 / EXP_02:

| Stage | Surface | Wall time | Cost | Gate |
|---|---|---|---|---|
| **A** Smoke | 50 stratified rows | ~1–2 min | $0 (Groq free tier) | always on |
| **B** Golden | 234 accepted golden rows | ~3–5 min | $0 | `RUN_GOLDEN = True` |
| **C** Test | **1,273 test-split rows** (locked 2026-05-06) | ~10 min | $0 | `RUN_FULL = False` (flip when A and B look right) |

Each stage writes `results/exp_03_sparse_rag__<surface>/` with `predictions.jsonl`, `retrieval.jsonl` (top-5 chunk_ids + raw BM25 scores per question), and `summary.json`. The runner is **resumable** — re-running any stage skips already-completed `question_id`s.

Tables this notebook fills: **Table 1** row 3 · **Table 8** row 2 (Retrieval Quality — Recall@K, MRR, nDCG; computed against golden 234) · **Table 9** Sparse-RAG column.

RAGAS metrics (all 5 — Faithfulness, Context Precision/Recall, Answer Relevancy, Answer Correctness) are filled by the companion notebook [`04c_exp03_ragas.ipynb`](04c_exp03_ragas.ipynb) after Stage B/C complete.

---

**The EXP_01 / EXP_02 baselines to compare against** (canonical test_1273):
- EXP_01 No-RAG: `Acuuracy = 0.7738`
- EXP_02 Naive RAG (dense): `Acuuracy = 0.7573` ← lost 1.65 pp because Context Precision was only 0.33

**Falsifiable hypothesis for EXP_03** (anchored in [`docs/output_notes/04b_exp02_output.md` §4](../docs/output_notes/04b_exp02_output.md)): *Sparse keyword matching will improve Context Precision on questions containing rare medical terms (drug names, anatomical structures, eponyms) where BM25 finds the literal term while BGE-large dense embedding gets distracted by semantically similar but topically wrong chunks.* Falsified if EXP_03 Context Precision ≤ 0.33.

## 1. Setup

In [1]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.data.indices import load_bm25
from src.data.loaders import load_chunks, load_golden, load_medqa_4opt
from src.eval.runner import run_experiment
from src.retrieval.sparse import SparseRetriever

print("GROQ_API_KEY present:", "\u2713" if os.environ.get("GROQ_API_KEY") else "\u2717")
print("Repo root:", REPO_ROOT)

GROQ_API_KEY present: ✓
Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project


## 2. Build the retriever (one-time, ~3 s)

Loads the BM25 pickle (built in Notebook 02; ~106 MB) + the chunks parquet for `chunk_id → text/book_name` lookup. The retriever is reused across all questions — don't re-create it per row.

In [2]:
BM25_PATH = REPO_ROOT / "data" / "indices" / "bm25.pkl"

chunks_df = load_chunks()
bm25_payload = load_bm25(BM25_PATH)
RETRIEVER = SparseRetriever(bm25_payload, chunks_df)

print(f"BM25 chunk_ids : {len(bm25_payload['chunk_ids']):,}")
print(f"chunks_df rows : {len(chunks_df):,}")

# Sanity check on a known-good rare-term query — BM25 should beat dense here
_test_chunks = RETRIEVER.retrieve(
    "What is the first-line treatment for community-acquired pneumonia?", k=3
)
print("\nSanity query (CAP first-line treatment) — top 3:")
for c in _test_chunks:
    print(f"  [{c.score:.2f}] {c.chunk_id} ({c.book_name}): {c.text[:90].replace(chr(10), ' ')}\u2026")

BM25 chunk_ids : 67,599
chunks_df rows : 67,599

Sanity query (CAP first-line treatment) — top 3:
  [38.16] InternalMed_Harrison_chunk_07112 (InternalMed_Harrison): Most significantly, M. pneumoniae is a major cause of community-acquired respiratory illne…
  [36.48] InternalMed_Harrison_chunk_06240 (InternalMed_Harrison): Aerosolization of Legionella by devices filled with tap water, including whirlpools, nebul…
  [35.16] Pharmacology_Katzung_chunk_02898 (Pharmacology_Katzung): Miscellaneous Antimicrobial Agents; Disinfectants, Antiseptics, & Sterilants  Camille E. B…


## 3. Configuration

In [8]:
EXPERIMENT_ID = "EXP_03_SPARSE_RAG"
RESULTS_DIR = REPO_ROOT / "results"
TOP_K = 5  # locked by plan.md §0 #12 (same as Naive)

SMOKE_N = 50
SMOKE_SEED = 42  # same seed as EXP_01/EXP_02 → directly comparable smoke surfaces

# Stage flags — flip these on once the previous stage looks right.
RUN_SMOKE = True       # Stage A · ~1–2 min
RUN_GOLDEN = True      # Stage B · ~3–5 min
RUN_FULL = True       # Stage C · ~10 min Groq on test split (1,273) — DON'T flip until A and B are clean

## 4. Stage A — Smoke (50 stratified rows)

Same stratified sample as EXP_01 / EXP_02 (seed 42 + `meta_info` weighting), so the smoke surfaces are directly comparable across architectures. BM25 is a pure-Python operation — expect ~5–10 ms retrieval per question, then Groq dominates wall time.

In [5]:
if RUN_SMOKE:
    medqa = load_medqa_4opt()
    smoke = (
        medqa.groupby("meta_info", group_keys=False)
        .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))
        .reset_index(drop=True)
    )
    smoke = smoke.head(SMOKE_N) if len(smoke) >= SMOKE_N else medqa.sample(n=SMOKE_N, random_state=SMOKE_SEED).reset_index(drop=True)

    print(f"Smoke surface: {len(smoke)} rows | meta_info mix: {dict(smoke['meta_info'].value_counts())}")

    smoke_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=smoke,
        output_dir=RESULTS_DIR / "exp_03_sparse_rag__smoke_50",
        experiment_id=EXPERIMENT_ID,
        dataset_label="smoke_50",
        k=TOP_K,
    )
    print(json.dumps(smoke_summary, indent=2))
else:
    print("RUN_SMOKE = False — skipping Stage A")

Smoke surface: 50 rows | meta_info mix: {'step1': np.int64(28), 'step2&3': np.int64(22)}


/var/folders/q2/0276zgxs0m7bj83vqzpkxlqh0000gn/T/ipykernel_59408/1883643334.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))


EXP_03_SPARSE_RAG:   0%|          | 0/50 [00:00<?, ?it/s]

{
  "experiment_id": "EXP_03_SPARSE_RAG",
  "dataset": "smoke_50",
  "n_questions": 50,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.9,
  "Exact_Match": 0.9,
  "n_correct": 45,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.4564075422286987,
  "wall_time_s_this_run": 174.748113155365,
  "n_calls_this_run": 50,
  "cache_hits_this_run": 0,
  "cache_hit_rate_this_run": 0.0,
  "timestamp_utc": "2026-05-07T10:56:29Z"
}


**Acceptance gate (Stage A):**
- `n_questions` == 50
- `Acuuracy` ≥ 0.65 (a sparse retriever may be coarser than dense; some pull from compute-budget — but smoke must clear 65 % to validate the wiring)
- `mean_latency_s` < 5.0 (BM25 retrieval is fast; Groq dominates)
- Spot-check `retrieval.jsonl` — every row should have **5 `retrieved_chunk_ids`** with **non-zero `retrieved_chunk_scores`** (raw BM25 scores typically 30–200+ for medical-question keyword overlap)

## 5. Stage B — Golden 234

In [7]:
if RUN_GOLDEN:
    golden = load_golden()
    print(f"Golden surface: {len(golden)} accepted rows")

    golden_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=golden,
        output_dir=RESULTS_DIR / "exp_03_sparse_rag__golden_234",
        experiment_id=EXPERIMENT_ID,
        dataset_label="golden_234",
        k=TOP_K,
    )
    print(json.dumps(golden_summary, indent=2))
else:
    print("RUN_GOLDEN = False — skipping Stage B")

Golden surface: 234 accepted rows
[runner] resuming: 190 of 234 already done


EXP_03_SPARSE_RAG:   0%|          | 0/234 [00:00<?, ?it/s]

{
  "experiment_id": "EXP_03_SPARSE_RAG",
  "dataset": "golden_234",
  "n_questions": 234,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.8376068376068376,
  "Exact_Match": 0.8376068376068376,
  "n_correct": 196,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.44850254670167583,
  "wall_time_s_this_run": 157.71063899993896,
  "n_calls_this_run": 44,
  "cache_hits_this_run": 0,
  "cache_hit_rate_this_run": 0.0,
  "timestamp_utc": "2026-05-07T11:18:18Z"
}


**Acceptance gate (Stage B):** `n_questions` == 234. EXP_01 golden_234 was 0.902, EXP_02 was 0.850. Sparse RAG accuracy could go either way — BM25 may help on rare-term questions (the falsifiable hypothesis) but hurt on questions where dense semantic matching was important. The interesting comparison is the RAGAS Context Precision (run via `04c_exp03_ragas.ipynb` after this stage).

## 6. Stage C — Test split (1,273 rows · gated · ~10 min)

**Set `RUN_FULL = True` in the config cell only after Stages A and B look right.** This is the canonical headline run — evaluation surface narrowed to the MedQA US `test` split per [`plan.md` §0 #8](../plan.md) (locked 2026-05-06).

BM25 retrieval is fast — wall time should be similar to or slightly less than EXP_02's ~10 min. The runner is resumable, so a mid-run failure (rate limit, network hiccup) just means re-running.

In [9]:
if RUN_FULL:
    medqa = load_medqa_4opt()
    medqa = medqa[medqa["split"] == "test"].reset_index(drop=True)
    print(f"Test-split surface: {len(medqa)} rows")

    full_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=medqa,
        output_dir=RESULTS_DIR / "exp_03_sparse_rag__test_1273",
        experiment_id=EXPERIMENT_ID,
        dataset_label="test_1273",
        k=TOP_K,
    )
    print(json.dumps(full_summary, indent=2))
else:
    print("RUN_FULL = False — skipping Stage C (set RUN_FULL = True in the config cell when ready)")

Test-split surface: 1273 rows


EXP_03_SPARSE_RAG:   0%|          | 0/1273 [00:00<?, ?it/s]

{
  "experiment_id": "EXP_03_SPARSE_RAG",
  "dataset": "test_1273",
  "n_questions": 1273,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.7580518460329929,
  "Exact_Match": 0.7580518460329929,
  "n_correct": 965,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.43538607917989414,
  "wall_time_s_this_run": 5827.474459171295,
  "n_calls_this_run": 1273,
  "cache_hits_this_run": 22,
  "cache_hit_rate_this_run": 0.01728201099764336,
  "timestamp_utc": "2026-05-07T12:55:49Z"
}


## 7. Inspect predictions — split-stratified comparison vs EXP_01 + EXP_02

The headline thesis question: did sparse retrieval help on the contamination-clean test split? Compare:
- **EXP_01 No-RAG**: 0.7738 on test_1273
- **EXP_02 Naive RAG (dense)**: 0.7573 on test_1273
- **EXP_03 Sparse RAG (BM25)**: ?? — fill in below

In [10]:
import pandas as pd

for label in ("smoke_50", "golden_234", "test_1273"):
    p = RESULTS_DIR / f"exp_03_sparse_rag__{label}" / "predictions.jsonl"
    if not p.exists():
        continue
    preds = pd.DataFrame(json.loads(line) for line in p.read_text().splitlines())
    print(f"\n=== {label}  (n={len(preds)}) ===")
    print(f"  overall accuracy : {preds['is_correct'].mean():.4f}")
    print(f"  mean latency     : {preds['latency_s'].mean():.3f} s")
    print(f"  cache hit rate   : {preds['was_cached'].mean()*100:.1f}%")

    if label == "test_1273":
        medqa = load_medqa_4opt().reset_index(drop=True)
        joined = preds.merge(medqa[["question_id", "meta_info", "split"]], on="question_id", how="left")
        print("  by meta_info:")
        for mi, g in joined.groupby("meta_info"):
            print(f"    {mi:10s}: n={len(g):5d}  acc={g['is_correct'].mean():.4f}")


=== smoke_50  (n=50) ===
  overall accuracy : 0.9000
  mean latency     : 0.456 s
  cache hit rate   : 0.0%

=== golden_234  (n=234) ===
  overall accuracy : 0.8376
  mean latency     : 0.449 s
  cache hit rate   : 0.4%

=== test_1273  (n=1273) ===
  overall accuracy : 0.7581
  mean latency     : 0.435 s
  cache hit rate   : 1.7%
  by meta_info:
    step1     : n=  679  acc=0.7452
    step2&3   : n=  594  acc=0.7727


## 8. Spot-check retrieval — do the chunks contain the literal keywords?

Eyeball the first 3 questions and their retrieved chunks. Sparse retrieval should match question keywords *literally* in the chunk text — if you don't see overlap, BM25 wiring is broken.

In [11]:
smoke_ret_path = RESULTS_DIR / "exp_03_sparse_rag__smoke_50" / "retrieval.jsonl"
smoke_pred_path = RESULTS_DIR / "exp_03_sparse_rag__smoke_50" / "predictions.jsonl"

if smoke_ret_path.exists() and smoke_pred_path.exists():
    rets = pd.DataFrame(json.loads(line) for line in smoke_ret_path.read_text().splitlines())
    preds = pd.DataFrame(json.loads(line) for line in smoke_pred_path.read_text().splitlines())
    medqa = load_medqa_4opt()
    medqa_q_by_qid = dict(zip(medqa["question_id"], medqa["question"]))

    chunk_text_by_id = dict(zip(chunks_df.chunk_id, chunks_df.text))

    for i in range(min(3, len(rets))):
        ret_row = rets.iloc[i]
        pred_row = preds.iloc[i]
        qid = ret_row["question_id"]
        print(f"\n=== {qid}  pred={pred_row['pred_letter']}  gold={pred_row['gold_letter']}  correct={pred_row['is_correct']} ===")
        print(f"Q: {medqa_q_by_qid.get(qid, '(not joined)')[:160]}\u2026")
        for cid, score in zip(ret_row["retrieved_chunk_ids"], ret_row["retrieved_chunk_scores"]):
            text = chunk_text_by_id.get(cid, "(missing)")
            print(f"  [{score:.2f}] {cid}: {text[:100].replace(chr(10), ' ')}\u2026")
else:
    print("No smoke retrieval/prediction files yet — run Stage A first.")


=== medqa_09246  pred=D  gold=D  correct=True ===
Q: A 25-year-old woman first presented to your clinic due to morning stiffness, symmetrical arthralgia in her wrist joints, and fatigue. She had a blood pressure o…
  [211.57] Biochemistry_Lippincott_chunk_01056: Case 4: Rapid Heart Rate, Headache, and Sweating  Patient Presentation: BE is a 45-year-old woman wh…
  [190.45] Pharmacology_Katzung_chunk_02193: Betty J. Dong, PharmD, FASHP, FCCP, FAPHA  JP is a 33-year-old woman who presents with complaints of…
  [187.30] Pharmacology_Katzung_chunk_03542: Drugs Used in the Treatment of Gastrointestinal Diseases  Kenneth R. McQuaid, MD  A 21-year-old woma…
  [182.19] Biochemistry_Lippincott_chunk_00684: Choose the ONE best answer. For Questions 26.1 and 26.2, use the following scenario.  A 40-year-old …
  [178.46] Biochemistry_Lippincott_chunk_00617: 3.2. In which one of the following tissues is glucose transport into the cell insulin dependent?  A.…

=== medqa_12184  pred=C  gold=C  correc

---

**Next.** With EXP_03 baseline complete, run [`04c_exp03_ragas.ipynb`](04c_exp03_ragas.ipynb) to score the golden 234 on **all 5 RAGAS metrics** using Claude Sonnet 4.6. **Critical comparison**: EXP_03 Context Precision vs EXP_02's 0.33 — if Sparse beats Naive on Context Precision, the falsifiable hypothesis from [`docs/output_notes/04b_exp02_output.md` §4](../docs/output_notes/04b_exp02_output.md) is supported. Cost ~$11–12, wall time ~10–20 min.